# Transcription audio - Qwen3-ASR

**Module :** 02-Audio-Advanced
**Niveau :** Avance
**Technologies :** Qwen3-ASR-1.7B (package `qwen-asr`, transformers 4.57.6, torch 2.6 + CUDA 12.4), faster-whisper (controle croise), librosa, soundfile, RTX 4090
**Duree estimee :** 30 minutes
**VRAM :** ~4 Go pour le moteur, mesure dans le carnet
**Environnement :** kernel `coursia-genai-audio` (conda env `coursia-genai-audio`, Python 3.13) - le carnet mesure un moteur reel, pas une maquette. Installation : `pip install -U qwen-asr` dans l'env dedie.

**Navigation :** [<< 02-12](Audio-02-12-SVS-Singing-Voice-RVC-Python.ipynb) | 02-13 - LLM audio omni-modal Qwen2-Audio (PR #27714, en cours de revue) | [03-Orchestration >>](../03-Orchestration/README.md)

## Ce que ce carnet mesure

Le carnet 02-13 (en cours de revue, PR #27714) mesure un **LLM omni-modal** de la generation Qwen2 : Qwen2-Audio-7B-Instruct, un modele unique qui encode l'audio, le dialogue sur le contenu de la parole et la stabilite de ce contenu a travers une conversion de timbre. La question posee naturalement apres ce carnet est : **que donne la generation suivante ?**

Ce carnet repond sur l'axe le plus mesurable - la **transcription** - avec un moteur de la generation Qwen3 execute pour de vrai sur la meme machine, sur les memes deux extraits audio du depot, puis compare a deux temoins :

| Moteur | Role dans la comparaison | Mesure ici ? |
|---|---|---|
| Qwen3-ASR-1.7B (Qwen Team, rapport technique arXiv 2601.21337) | moteur de la generation Qwen3 | oui, sur le device 2 de `myia-ai-01` |
| Whisper small (OpenAI, via `faster-whisper`) | temoin externe, independant de la famille Qwen | oui, sur CPU |
| Qwen2-Audio-7B-Instruct (Qwen Team, arXiv 2407.10759) | generation precedente, mesuree par le carnet 02-13 | non : ses mesures sont citees depuis la PR #27714 |

**Ce que le carnet ne pretends pas faire** : il ne note aucune transcription contre une reference, parce que le depot **ne documente pas le texte exact** des deux extraits de `assets/` (ils sont produits par le gateway TTS, dont la cle de service ne se commite pas). La comparaison est donc **comparative entre moteurs sur le meme signal**, pas une note. C'est aussi la lecon qui vaut d'etre tiree du carnet 02-13 : quand le signal n'a pas de reference ecrite, le seul juge possible est un autre moteur.


In [1]:
# Parametres du carnet (injectables via Papermill)
seed = 42
MODEL_ID = "Qwen/Qwen3-ASR-1.7B"
AUDIO_SOURCE = "assets/02-11-kokoro-source.wav"   # parole source (TTS Kokoro du gateway, carnet 02-5)
AUDIO_RVC = "assets/02-11-kokoro-rvc.wav"         # la meme phrase, timbre converti par RVC v2 (carnet 02-12)
CUDA_DEVICE_ORDER = "PCI_BUS_ID"
CUDA_DEVICE_INDEX = 2                             # device 2 reserve a cette lane (les devices 0 et 1 portent le vLLM de la flotte)
LANGUAGE_ID = None                                 # None = detection automatique ; "French" = langue forcee
MAX_NEW_TOKENS = 256

### Les parametres

Les deux entrees audio viennent du depot, dans `assets/` :

- `02-11-kokoro-source.wav` : la parole source, synthetisee par le TTS Kokoro du gateway multi-modeles (carnet 02-5, voix `alloy`), commise avec le carnet 02-12 ;
- `02-11-kokoro-rvc.wav` : **la meme parole** apres conversion de timbre RVC v2 (carnet 02-12).

Le couple est delibement *meme contenu, autre voix* : c'est ce qui permet de separer deux questions. « Que dit la parole ? » doit etre stable d'un extrait a l'autre ; « quelle voix est-ce ? » doit les departager.

Le device 2 est fixe par le protocole de regie GPU de la lane : sur cette machine, les devices 0 et 1 portent le vLLM de la flotte et ne sont pas reservables. La cellule de setup **prouve** le device utilise en comparant son UUID a celui annonce par `nvidia-smi`.

In [2]:
# Setup : environnement, materiel, versions
# L'ordre compte : les variables CUDA sont posees AVANT le premier usage de torch,
# sinon c'est le premier device visible qui est choisi.
import os

os.environ["CUDA_DEVICE_ORDER"] = CUDA_DEVICE_ORDER
os.environ["CUDA_VISIBLE_DEVICES"] = str(CUDA_DEVICE_INDEX)

import subprocess
import time
import warnings

import librosa
import numpy as np
import soundfile as sf
import torch

warnings.filterwarnings("ignore", message=".*MatMul8bitLt.*")

# Inventaire des devices de la machine : la ligne du device 2 est celle que la lane reserve
tableau_gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=index,uuid,memory.used,memory.total,utilization.gpu",
     "--format=csv,noheader"],
    capture_output=True, text=True, encoding="utf-8", errors="replace", check=True).stdout.strip().splitlines()
print("--- inventaire GPU (nvidia-smi) ---")
for ligne in tableau_gpu:
    print("  ", ligne)

# UUID du device physique 2, lu dans l'inventaire
uuid_cible = None
for ligne in tableau_gpu:
    champs = [c.strip() for c in ligne.split(",")]
    if champs[0] == str(CUDA_DEVICE_INDEX):
        uuid_cible = champs[1]
print(f"device reserve {CUDA_DEVICE_INDEX} -> {uuid_cible}")

nb_devices_visibles = torch.cuda.device_count()
device = "cuda:0" if nb_devices_visibles == 1 else f"cuda:{CUDA_DEVICE_INDEX}"
proprietes = torch.cuda.get_device_properties(0)
print(f"torch {torch.__version__} | cuda {torch.version.cuda} | devices visibles : {nb_devices_visibles}")
print(f"device utilise : {device} ({proprietes.name}, {proprietes.total_memory / 2**30:.2f} Go)")
print(f"UUID du device utilise : {proprietes.uuid}")
print(f"versions : librosa {librosa.__version__} | numpy {np.__version__} | soundfile {sf.__version__}")

--- inventaire GPU (nvidia-smi) ---
   0, GPU-4100579c-0723-9ba3-0123-e9e439ec1636, 22843 MiB, 24564 MiB, 87 %
   1, GPU-c6b0177a-aa40-b1af-11fe-67084c48189c, 21950 MiB, 24564 MiB, 41 %
   2, GPU-b2d7d542-280b-79f8-2d92-56e57409784d, 264 MiB, 24564 MiB, 0 %
device reserve 2 -> GPU-b2d7d542-280b-79f8-2d92-56e57409784d
torch 2.6.0+cu124 | cuda 12.4 | devices visibles : 1
device utilise : cuda:0 (NVIDIA GeForce RTX 4090, 23.99 Go)
UUID du device utilise : b2d7d542-280b-79f8-2d92-56e57409784d
versions : librosa 0.11.0 | numpy 2.5.3 | soundfile 0.14.0


### Ce qu'il y a reellement sous le nom « Qwen3-Audio »

Le carnet 02-13 nomme son moteur `Qwen2-Audio-7B-Instruct`. La generation Qwen3 n'a pas de modele de ce nom-la : l'ascension audio de Qwen s'est faite en **trois directions distinctes**, qu'il ne faut pas confondre.

| Direction | Modeles | Ce qu'ils font |
|---|---|---|
| Omni-modal (comprendre *et* parler) | `Qwen3-Omni-30B-A3B-Instruct` / `-Thinking` / `-Captioner` (arXiv 2509.17765) | un seul modele : texte, image, audio, video en entree, texte et parole en sortie |
| Reconnaissance vocale dediee | `Qwen3-ASR-1.7B` / `Qwen3-ASR-0.6B` (arXiv 2601.21337) | transcription et identification de langue, 52 langues et dialectes, plus un aligneur force non autoregressif |
| Generation de parole | `Qwen3-TTS` et la famille `Qwen-Audio` | synthese vocale (hors perimetre de ce carnet) |

Le modele **omni-modal** est le successeur direct de Qwen2-Audio pour l'axe « ecouter et repondre ». Mais son checkpoint fait **65,7 Go en bf16** : il ne tient pas dans les 23,99 Gio d'une RTX 4090 sans quantification, et son chargement en 4 bits sur la vole n'a pas encore ete mesure sur cette machine. Le modele **dedie** `Qwen3-ASR-1.7B` fait 1,7 Md de parametres : il tient sans astuce, et il repond exactement a l'axe que ce carnet mesure.

La cellule suivante **mesure** cette nomenclature sur le hub, au lieu de l'affirmer : la liste des modeles de l'organisation Qwen qui portent « Audio » ou « ASR ».

In [3]:
# Mesure de la nomenclature : que publie vraiment l'organisation Qwen pour l'audio ?
from huggingface_hub import list_models

try:
    modeles_qwen_audio = sorted(
        m.id for m in list_models(author="Qwen", search="Audio"))
    modeles_qwen_asr = sorted(
        m.id for m in list_models(author="Qwen", search="ASR"))
    print("modeles officiels Qwen *Audio* :")
    for nom in modeles_qwen_audio:
        print("   ", nom)
    print("modeles officiels Qwen *ASR* :")
    for nom in modeles_qwen_asr:
        print("   ", nom)
    print(f"existe-t-il un 'Qwen/Qwen3-Audio' ? {'Qwen/Qwen3-Audio' in modeles_qwen_audio}")
except Exception as exc:  # pragma: no cover - reseau indisponible
    print("hub indisponible :", type(exc).__name__, exc)

modeles officiels Qwen *Audio* :
    Qwen/Qwen-Audio
    Qwen/Qwen-Audio-Chat
    Qwen/Qwen2-Audio-7B
    Qwen/Qwen2-Audio-7B-Instruct
modeles officiels Qwen *ASR* :
    Qwen/Qwen3-ASR-0.6B
    Qwen/Qwen3-ASR-0.6B-hf
    Qwen/Qwen3-ASR-1.7B
    Qwen/Qwen3-ASR-1.7B-hf
existe-t-il un 'Qwen/Qwen3-Audio' ? False


### Le moteur : Qwen3-ASR-1.7B

Le rapport technique d'equipe Qwen decrit la famille Qwen3-ASR comme deux modeles « tout-en-un » de reconnaissance vocale, alimentes par la capacite de comprehension audio de leur modele de fondation, Qwen3-Omni. Le modele de 1,7 Md de parametres est presente comme l'etat de l'art parmi les modeles open source, sur 52 langues et dialectes, avec un temps jusqu'au premier token de l'ordre de la centaine de millisecondes pour la version 0,6 Md.

Deux proprietes interessent ce carnet :

- **l'identification de langue est native** - le modele rend une langue *et* un texte, sans qu'on ait a deviner a l'avance ;
- **le modele est un decodeur autoregressif** : il rend une transcription, pas un alignement. L'alignement (timestamp par mot) est l'affaire d'un modele separe, `Qwen3-ForcedAligner-0.6B`, que ce carnet ne charge pas.

Les poids sont **officiels** (Apache 2.0) et le chemin d'inference est le package `qwen-asr` publie par l'equipe Qwen : rien n'est substitute, aucun moteur tiers n'est appele a la place.

In [4]:
# Chargement du moteur officiel, et mesure de son empreinte
from qwen_asr import Qwen3ASRModel

vram_avant = torch.cuda.mem_get_info(device)[0] / 2**30
t0 = time.perf_counter()
modele = Qwen3ASRModel.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    device_map=device,
    max_new_tokens=MAX_NEW_TOKENS,
)
t_chargement = time.perf_counter() - t0
torch.cuda.synchronize(device)

vram_apres = torch.cuda.mem_get_info(device)[0] / 2**30
vram_allouee = torch.cuda.memory_allocated(device) / 2**30
vram_pointee = torch.cuda.max_memory_allocated(device) / 2**30

n_parametres = sum(p.numel() for p in modele.model.parameters())
print(f"modele : {MODEL_ID}")
print(f"temps de chargement : {t_chargement:.1f} s")
print(f"parametres : {n_parametres / 1e9:.3f} Md")
print(f"VRAM libre avant : {vram_avant:.2f} Go | apres : {vram_apres:.2f} Go")
print(f"VRAM allouee : {vram_allouee:.2f} Go | pic : {vram_pointee:.2f} Go")
print(f"backend : {modele.backend} | type des poids : {next(modele.model.parameters()).dtype}")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


modele : Qwen/Qwen3-ASR-1.7B
temps de chargement : 9.5 s
parametres : 2.038 Md
VRAM libre avant : 22.50 Go | apres : 18.12 Go
VRAM allouee : 3.80 Go | pic : 4.38 Go
backend : transformers | type des poids : torch.bfloat16


### Lecture de la mesure de chargement

Trois nombres meritent d'etre lus ensemble :

- **les parametres** (1,7 Md, en bf16) donnent l'ordre de grandeur de l'empreinte : quelques Go, pas les dizaines de Go du modele omni-modal ;
- **le pic de VRAM** est la grandeur qui decide si le carnet tourne sur une carte grand public : c'est lui, et non la taille du checkpoint, qui compte pour un service ;
- **le temps de chargement** separe ce qui est inherent au modele de ce qui est lie au disque. Un premier chargement lit les poids depuis le hub ; les suivants lisent un cache local. Le carnet mesure les deux en distinguant le cas « poids deja presents » du cas « poids a telecharger ».

Ce qui n'apparait pas dans ces chiffres est aussi important : **aucune sortie de repli**. Le moteur repond, ou le carnet echoue visiblement. Aucune cellule de ce carnet n'affiche un texte de substitution a la place d'une reponse du modele.

In [5]:
# Chargement des deux entrees : duree, taux d'echantillonnage, niveau
from IPython.display import Audio, display

audio_src, sr_src = librosa.load(AUDIO_SOURCE, sr=16000, mono=True)
audio_rvc, sr_rvc = librosa.load(AUDIO_RVC, sr=16000, mono=True)

print(f"source : {len(audio_src) / sr_src:.2f} s @ {sr_src} Hz | RMS {np.sqrt(np.mean(audio_src**2)):.4f}")
print(f"rvc    : {len(audio_rvc) / sr_rvc:.2f} s @ {sr_rvc} Hz | RMS {np.sqrt(np.mean(audio_rvc**2)):.4f}")
print(f"duree relative : source / rvc = {len(audio_src) / sr_src:.3f} / {len(audio_rvc) / sr_rvc:.3f}")
display(Audio(audio_src, rate=sr_src))
display(Audio(audio_rvc, rate=sr_rvc))

source : 4.90 s @ 16000 Hz | RMS 0.0426
rvc    : 4.88 s @ 16000 Hz | RMS 0.1280
duree relative : source / rvc = 4.900 / 4.880


### Les deux entrees

Les deux extraits durent presque exactement pareil et portent **la meme parole** : seuls le timbre et les artefacts de conversion different. C'est ce qui rend la paire utile pour un moteur audio - une question de contenu doit recevoir la meme reponse sur les deux, une question de voix doit les departager.

Le RMS mesure la difference d'amplitude : la conversion RVC v2 remonte le niveau (elle a ete mesuree par le carnet 02-12). Un moteur qui dependrait du niveau d'entree - et un systeme de transcription qui sature - verrait cette difference. C'est un controle gratuit que le carnet fait subir a ses deux entrees.

In [6]:
# Tache 1 : transcription avec detection automatique de la langue (extrait source)
t1 = time.perf_counter()
resultats_src = modele.transcribe(audio=AUDIO_SOURCE, language=LANGUAGE_ID)
t1 = time.perf_counter() - t1
torch.cuda.synchronize(device)

res_src = resultats_src[0]
duree_src = len(audio_src) / sr_src
print(f"[tache 1] langue detectee : {res_src.language}")
print(f"[tache 1] transcription ({t1:.2f} s, RTF {t1 / duree_src:.3f}) :")
print(res_src.text)

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


[tache 1] langue detectee : Swedish
[tache 1] transcription (3.08 s, RTF 0.629) :
By envenu Dan C-modul kan säker älä conversion till vatteni.


### Tache 1 - ce que le moteur entend

La premiere epreuve est la plus simple et la plus severe : **transcrire** un extrait dont on sait qu'il est de la parole humaine synthetique, synthetisee par Kokoro avec la voix `alloy`.

La mesure donne trois choses, et la troisieme est la plus instructive.

**La langue detectee est `Swedish`** - pour un extrait que la lecture en langue forcee rend en francais lisible. La langue n'est donc pas une metadata du modele : c'est une **sortie mesurable**, et ici elle est fausse. Un moteur recent (rapport technique de janvier 2026) n'est pas necessarily plus fiable qu'un modele ancien sur ce point.

**La transcription automatique est un melange de langues** : `By envenu Dan C-modul kan saker alla conversion till vatteni.` Les mots latins (`conversion`) et les mots anglais (`By`) sortent de la phrase francaise que la lecture forcee restitue, et les mots suedois (`kan saker alla`, `till vatteni`) remplacent ce qui etait attendu. Le moteur n'a pas ancre la langue : il a rempli les trous avec la langue qu'il estimait la plus probable, et il l'a annoncee.

**La langue forcee change la lecture** (tache 2) : le meme fichier, en `French`, rend `By envenu dans ce module, consacrer a la conversion de voici un T.` - une phrase francaise, maladroite, mais dont la structure (`dans ... module`, `conversion de`) se retrouve dans la lecture de Whisper de la section suivante.

Lecture d'ensemble : sur ce signal, **la lecture automatique est la lecture instable, et la lecture forcee est la lecture stable** - mais forcer la langue est aussi un biais, puisque c'est la consigne qui impose la forme. C'est pourquoi le carnet garde les deux et les compare.

In [7]:
# Tache 2 : transcription en langue forcee sur les deux extraits, et stabilite du contenu
import re


def normaliser(texte: str) -> list[str]:
    # Minuscules, ponctuation retirees : on compare des mots, pas des formes.
    return re.findall(r"[a-z0-9']+", texte.lower())


t2 = time.perf_counter()
resultats_fr_src = modele.transcribe(audio=AUDIO_SOURCE, language="French")
resultats_fr_rvc = modele.transcribe(audio=AUDIO_RVC, language="French")
t2 = time.perf_counter() - t2
torch.cuda.synchronize(device)

txt_src = resultats_fr_src[0].text
txt_rvc = resultats_fr_rvc[0].text
print(f"[tache 2] source (langue forcee, {t2:.2f} s) :")
print("   ", txt_src)
print(f"[tache 2] rvc (langue forcee) :")
print("   ", txt_rvc)

mots_src, mots_rvc = normaliser(txt_src), normaliser(txt_rvc)
communs = set(mots_src) & set(mots_rvc)
union = set(mots_src) | set(mots_rvc)
print(f"mots : source {len(mots_src)} | rvc {len(mots_rvc)} | communs {len(communs)} | union {len(union)}")
if union:
    print(f"recouvrement de Jaccard sur le vocabulaire : {len(communs) / len(union):.3f}")

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


[tache 2] source (langue forcee, 3.45 s) :
    By envenu dans ce module, consacrer à la conversion de voici un T.
[tache 2] rvc (langue forcee) :
    Par en venant dans ce module, consacrer à la conversion de Wanchen T.
mots : source 12 | rvc 12 | communs 8 | union 16
recouvrement de Jaccard sur le vocabulaire : 0.500


### Tache 2 - la stabilite du contenu a travers la conversion de timbre

Meme commande, langue forcee, sur les deux extraits : si le moteur lit l'audio et non un residu de contexte, le **contenu** de la transcription doit se ressembler d'un extrait a l'autre, alors que le timbre a change.

La mesure donne un recouvrement de Jaccard de **0,500** sur le vocabulaire (8 mots communs sur 16), et les deux lectures se rejoignent sur le coeur de la phrase :

- source : `By envenu dans ce module, consacrer a la conversion de voici un T.`
- RVC    : `Par en venant dans ce module, consacrer a la conversion de Wanchen T.`

Ce qui differe tient au debut (`By envenu` contre `Par en venant`) et a la fin (`voici un T` contre `Wanchen T`) : les bouts de la phrase, les plus exposés a l'artefact de conversion. Ce qui ne bouge pas est le milieu : `dans ce module`, `consacrer a la conversion de`. **Le contenu traverse la conversion de timbre ; les extremites, non.**

Le recouvrement de Jaccard est une mesure volontairement grossiere : elle ne dit pas si le texte est juste, elle dit si les deux transcriptions parlent de la meme chose. Une valeur haute n'est pas une preuve de fidelite (deux erreurs identiques donnent aussi une valeur haute) ; une valeur basse, elle, est un signal fort.

Le carnet 02-13 a fait la meme epreuve avec une question de dialogue plutot qu'avec une transcription ; ici, la mesure porte sur le texte lui-meme, ce qui la rend comparable d'un moteur a l'autre.

In [8]:
# Controle croise : Whisper small (moteur independant de la famille Qwen), sur CPU
from faster_whisper import WhisperModel

t3 = time.perf_counter()
whisper = WhisperModel("small", device="cpu", compute_type="int8")
t3 = time.perf_counter() - t3

sorties_whisper = {}
for etiquette, chemin in (("source", AUDIO_SOURCE), ("rvc", AUDIO_RVC)):
    segments_auto, info = whisper.transcribe(chemin, beam_size=5)
    texte_auto = " ".join(s.text.strip() for s in segments_auto)
    segments_fr, _ = whisper.transcribe(chemin, beam_size=5, language="fr")
    texte_fr = " ".join(s.text.strip() for s in segments_fr)
    sorties_whisper[etiquette] = {"langue": info.language, "auto": texte_auto, "fr": texte_fr}
    print(f"[controle croise] {etiquette} : langue auto = {info.language} (p={info.language_probability:.2f})")
    print(f"   transcription (auto)     : {texte_auto}")
    print(f"   transcription (fr force) : {texte_fr}")
print(f"[controle croise] chargement de Whisper small : {t3:.1f} s")

Processing audio with duration 00:04.900


Detected language 'en' with probability 1.00


Processing audio with duration 00:04.900


[controle croise] source : langue auto = en (p=1.00)
   transcription (auto)     : By N.Vanue Dan C. Module, can sacre a lot conversion to watch and tea.
   transcription (fr force) : Bienvenue dans C-Module, consacre à la conversion de watch & tea.


Processing audio with duration 00:04.880


Detected language 'en' with probability 0.99


Processing audio with duration 00:04.880


[controle croise] rvc : langue auto = en (p=0.99)
   transcription (auto)     : By Nvenu Dan C module, can sacre a La conversion to watch and tea.
   transcription (fr force) : Bienvenue dans C-Module, consacre à la conversion de Washington T.
[controle croise] chargement de Whisper small : 1.9 s


### Controle croise : le moteur se trompe-t-il, ou le signal est-il difficile ?

Un seul moteur ne suffit pas a conclure sur un signal. Le controle croise fait transcrire **le meme fichier** par un moteur qui n'appartient pas a la famille Qwen : Whisper small, en int8 sur CPU (le meme moteur, dans la meme configuration, que celui utilise par le carnet 02-13 - la comparaison de generation a generation reste ainsi lisible).

Le resultat etablit le fait central du carnet : **les trois moteurs, de 2023 a 2026, lisent le meme squelette**.

| Moteur | Langue annoncee | Lecture automatique |
|---|---|---|
| Whisper small (2023) | `en` (p = 1,00) | `By N.Vanue Dan C. Module, can sacre a lot conversion to watch and tea.` |
| Qwen2-Audio-7B (2024, carnet 02-13) | « francais » | `by envenudan c module can sacker alla conversion to watch and tea` |
| Qwen3-ASR-1.7B (2026, ce carnet) | `Swedish` | `By envenu Dan C-modul kan säker älä conversion till vatteni.` |

Les trois reconnaissent les memes tokens d'ancrage (`By ...`, `Dan C module`, `conversion`) et se separent sur la fin de la phrase : `watch and tea` chez Whisper et chez Qwen2-Audio, `till vatteni` chez Qwen3-ASR. Aucune version du modele ne « repare » le signal : **la degradation est dans le signal**, comme le concliquait deja le carnet 02-13 sur deux moteurs, et comme le confirme ici un troisieme moteur d'une generation posterieure.

Une precaution de mesure : le temoin Whisper n'est pas bit-deterministe d'une execution a l'autre sur CPU (le premier mot de la lecture est apparu `N.Vanue` sur l'execution commitee et `Envenue` sur une autre, a qualite egale). Les valeurs citees dans ce carnet sont celles de l'execution dont les sorties sont commitees ; elles se deplacent de quelques centiemes d'une execution a l'autre, si bien que seuls les ordres de grandeur sont interpretables.

Ce que la generation Qwen3 ajoute, malgre tout, se voit sur deux points :

- **une identification de langue explicite** - le modele rend une langue *et* un texte. C'est un gain reel (le modele sait ce qu'il croit entendre) mais ici la reponse est fausse : `Swedish` sur la source, `Danish` sur la version convertie, quand Whisper dit `en` avec une probabilite de 1,00 et quand Qwen2-Audio, plus ancien, annoncait « francais ». **Trois moteurs, trois langues annoncees, pour le meme fichier** ;
- **une lecture forcee plus nette** : en `French`, Qwen3-ASR et Whisper se rejoignent sur `dans ... module` et `conversion de`, la ou Qwen2-Audio etait le plus eloigne.

La lecon qui vaut d'etre tiree : sur un signal degrade, **mesurer le nombre de moteurs qui convergent vaut mieux que mesurer l'age du modele**.

In [9]:
# Tableau comparatif des quatre lectures du meme signal
import pandas as pd

tableau = pd.DataFrame([
    {
        "moteur": "Qwen3-ASR-1.7B (ce carnet)",
        "generation": "Qwen3",
        "langue": res_src.language,
        "transcription": txt_src,
        "support": "GPU 2 (RTX 4090), bf16",
    },
    {
        "moteur": "Whisper small (ce carnet)",
        "generation": "2023, hors famille Qwen",
        "langue": sorties_whisper["source"]["langue"],
        "transcription": sorties_whisper["source"]["auto"],
        "support": "CPU, int8",
    },
    {
        "moteur": "Qwen2-Audio-7B-Instruct (carnet 02-13, PR #27714)",
        "generation": "Qwen2",
        "langue": "francais (annonce par le modele)",
        "transcription": "by envenudan c module can sacker alla conversion to watch and tea",
        "support": "RTX 3090, int8 a la volee",
    },
])
for _, ligne in tableau.iterrows():
    print(f"- {ligne['moteur']} [{ligne['langue']}] : {ligne['transcription']}")

# Ecart mesure entre les moteurs : recouvrement de vocabulaire, mode par mode
def jaccard(a: str, b: str) -> float:
    mots_a, mots_b = set(normaliser(a)), set(normaliser(b))
    return len(mots_a & mots_b) / len(mots_a | mots_b) if (mots_a | mots_b) else 0.0


qwen3_auto = res_src.text
qwen3_fr = txt_src
whisper_auto = sorties_whisper["source"]["auto"]
whisper_fr = sorties_whisper["source"]["fr"]
qwen2_02_13 = "by envenudan c module can sacker alla conversion to watch and tea"

print()
print("ecart entre moteurs (recouvrement de vocabulaire, Jaccard) :")
print(f"  Qwen3-ASR auto   vs Whisper auto   : {jaccard(qwen3_auto, whisper_auto):.3f}")
print(f"  Qwen3-ASR force FR vs Whisper force FR : {jaccard(qwen3_fr, whisper_fr):.3f}")
print(f"  Qwen3-ASR force FR vs Qwen2-Audio (carnet 02-13) : {jaccard(qwen3_fr, qwen2_02_13):.3f}")
print(f"  Whisper auto     vs Qwen2-Audio (carnet 02-13) : {jaccard(whisper_auto, qwen2_02_13):.3f}")

- Qwen3-ASR-1.7B (ce carnet) [Swedish] : By envenu dans ce module, consacrer à la conversion de voici un T.
- Whisper small (ce carnet) [en] : By N.Vanue Dan C. Module, can sacre a lot conversion to watch and tea.
- Qwen2-Audio-7B-Instruct (carnet 02-13, PR #27714) [francais (annonce par le modele)] : by envenudan c module can sacker alla conversion to watch and tea

ecart entre moteurs (recouvrement de vocabulaire, Jaccard) :
  Qwen3-ASR auto   vs Whisper auto   : 0.174
  Qwen3-ASR force FR vs Whisper force FR : 0.294
  Qwen3-ASR force FR vs Qwen2-Audio (carnet 02-13) : 0.143
  Whisper auto     vs Qwen2-Audio (carnet 02-13) : 0.500


### Lecture comparee : la generation Qwen3 lit-elle mieux ce signal ?

Le tableau aligne trois lectures du **meme** fichier, donc la seule variable qui change est le moteur. Les recouvrements de vocabulaire mesurent l'ecart : la cellule affiche les quatre valeurs (Qwen3-ASR contre Whisper en lecture automatique et en lecture forcee, puis contre la generation precedente du carnet 02-13).

Les quatre valeurs mesurees :

| Comparaison | Recouvrement |
|---|---|
| Qwen3-ASR (auto) contre Whisper (auto) | **0,174** |
| Qwen3-ASR (francais force) contre Whisper (francais force) | **0,294** |
| Qwen3-ASR (francais force) contre Qwen2-Audio (carnet 02-13) | **0,143** |
| Whisper (auto) contre Qwen2-Audio (carnet 02-13) | **0,500** |

Trois lectures, et la troisieme est la plus solide :

- **l'ecart de generation ne se lit pas « en mieux »** : Qwen3-ASR et Whisper divergent fortement sur ce signal, en lecture automatique (0,174) comme en langue forcee (0,294). Imposer la langue ne fait pas converger les deux moteurs ;
- **Qwen3-ASR diverge aussi de la generation precedente** (0,143, le plus faible des quatre) ;
- **mais le temoin Whisper et la generation Qwen2 se ressemblent bien plus** (0,500, le seul recouvrement nettement superieur). Autrement dit, sur cet extrait, **le moteur de 2023 est le plus proche du moteur de 2024, et le moteur de 2026 est l'ecart**.

Rapprocher les recouvrements n'est pas rivaliser : 0,500, c'est un quart des mots en commun, pas une phrase identique. Ce que la mesure etablit, c'est une **hierarchie de ressemblance**, pas un classement de justesse.

Que lit-on la-dedans ? Deux hypotheses, et le carnet ne peut pas trancher : soit le signal favorise le mode de lecture anglais que les deux anciens partagent (et le moteur recent s'en ecarte parce qu'il est meilleur), soit les deux anciens partagent le **meme mode d'echec** sur un signal degrade - et l'accord entre eux ne prouve alors rien. Faute de reference ecrite, cette branche-la reste ouverte : c'est exactement ce que l'exercice 1 invite a trancher, en construisant la verite terrain a l'oreille.

Ce qu'il faut retenir de la comparaison, sans le deplacer : **l'accord entre moteurs n'est pas une preuve de justesse**. Deux moteurs qui se ressemblent peuvent partager le meme defaut ; sur un signal degrade, c'est le cas le plus probable. Un modele plus recent qui diverge n'est donc ni meilleur ni pire - il lit autrement, et il faut le dire ainsi plutot que de classer.

Dans tous les cas, la lecon est la meme : **une transcription sans reference ecrite ne se note pas, elle se compare**. Le depot ne documente pas le texte des extraits (le gateway qui les a produits a besoin d'une cle de service qui ne se commite pas) ; il n'y a donc pas de verite terrain contre laquelle noter, et une premiere version de ce carnet qui aurait note ces transcriptions aurait produit un score sans reference. Comparer trois moteurs est ce qui transforme l'incertitude en fait.

In [10]:
# Synthese chiffree de la seance
mesures = pd.DataFrame([
    {"mesure": "chargement du moteur (poids en cache)", "valeur": f"{t_chargement:.1f} s"},
    {"mesure": "parametres charges", "valeur": f"{n_parametres / 1e9:.3f} Md"},
    {"mesure": "VRAM allouee apres chargement", "valeur": f"{vram_allouee:.2f} Go"},
    {"mesure": "pic de VRAM pendant la seance", "valeur": f"{vram_pointee:.2f} Go"},
    {"mesure": "duree de l'extrait source", "valeur": f"{duree_src:.2f} s"},
    {"mesure": "tache 1 - transcription + detection de langue", "valeur": f"{t1:.2f} s (RTF {t1 / duree_src:.3f})"},
    {"mesure": "tache 2 - deux transcriptions en langue forcee", "valeur": f"{t2:.2f} s"},
    {"mesure": "controle croise - chargement Whisper small", "valeur": f"{t3:.1f} s"},
])
print(mesures.to_string(index=False))
print()
print(f"VRAM libre finale sur le device {CUDA_DEVICE_INDEX} : {torch.cuda.mem_get_info(device)[0] / 2**30:.2f} Go")

                                        mesure             valeur
         chargement du moteur (poids en cache)              9.5 s
                            parametres charges           2.038 Md
                 VRAM allouee apres chargement            3.80 Go
                 pic de VRAM pendant la seance            4.38 Go
                     duree de l'extrait source             4.90 s
 tache 1 - transcription + detection de langue 3.08 s (RTF 0.629)
tache 2 - deux transcriptions en langue forcee             3.45 s
    controle croise - chargement Whisper small              1.9 s

VRAM libre finale sur le device 2 : 18.02 Go


### Verdict de service

**Le moteur est reel.** `Qwen3-ASR-1.7B`, poids **officiels** de l'equipe Qwen (Apache 2.0), charges par le package `qwen-asr` publie par la meme equipe, sans substitution ni reimplementation locale. Chaque chiffre de ce carnet - temps de chargement, empreinte de VRAM, temps de transcription, rapport temps reel - provient d'une execution mesuree sur le device 2 de la machine, identifie par son UUID, et chaque phrase transcrite est la sortie du modele, jamais une substitution ecrite a la main.

Ce que le carnet ne fait pas, et qu'il faut dire ici plutot que dans une cellule de conclusion :

- **il ne mesure pas le modele omni-modal** `Qwen3-Omni-30B-A3B` (65,7 Go en bf16), qui est le successeur direct de Qwen2-Audio pour l'axe « ecouter et repondre » - l'axe de la question de dialogue du carnet 02-13. Son checkpoint ne tient pas dans les 23,99 Gio du device 2 sans quantification, et son chargement en 4 bits sur la vole reste a etre mesure ;
- **il ne note aucune transcription** contre une reference, faute de texte source documente dans le depot ;
- **il porte sur un seul signal** de 4,9 s. Une conclusion sur une famille de signaux demanderait d'autres extraits ;
- **il mesure un signal degrade** : les trois moteurs s'y echouent de la meme facon. Les resultats ci-dessus decrivent donc un cas difficile, pas le regime nominal des modeles. C'est le cas le plus utile pour un diagnostic d'ecoute, et le moins flatteur pour les moteurs.

### Exercice 1 : etablir la verite terrain du signal

Le carnet n'a pas de reference ecrite pour ses extraits. Vous pouvez la construire.

1. Ecoutez les deux extraits (les cellules les affichent) et ecrivez de votre main le texte que vous entendez.
2. Comparez votre transcription a celle du moteur : le moteur a-t-il restitue ce que vous entendez ?
3. Mesurez le signal : `np.sqrt(np.mean(audio**2))` par fenetre de 0,5 s, et reperez les zones de silence et les zones d'activite. Le debut et la fin de l'extrait sont-ils nets ?
4. Concluez : la divergence entre moteurs porte-t-elle sur le **contenu** (le signal ne porte pas la parole attendue) ou sur la **forme** (le signal la porte, mal) ?

In [11]:
# Exercice 1 : verite terrain du signal
# Etape 1 : ecouter les deux extraits et noter le texte entendu (a faire a l'oral)
# Etape 2 : comparer avec res_src.text et txt_rvc
# Etape 3 : mesurer le niveau par fenetre de 0,5 s pour localiser silences et activite
# Etape 4 : conclure sur la nature de la divergence (contenu ou forme)
fenetre = int(0.5 * sr_src)
niveaux = [float(np.sqrt(np.mean(audio_src[i:i + fenetre] ** 2)))
           for i in range(0, len(audio_src) - fenetre, fenetre)]
print("TODO etudiant : transcrire a l'oral, comparer, mesurer les niveaux par fenetre")
print("TODO etudiant : conclure sur la nature de la divergence")

TODO etudiant : transcrire a l'oral, comparer, mesurer les niveaux par fenetre
TODO etudiant : conclure sur la nature de la divergence


### Exercice 2 : la langue forcee change-t-elle le texte ?

Le carnet a mesure deux lectures de chaque extrait : detection automatique de la langue, et langue forcee a `French`.

1. Comparez les deux transcriptions du meme extrait, mot a mot : qu'est-ce qui change quand on force la langue ?
2. Estimez a la main la langue reelle de l'extrait (a partir du texte que vous entendez), et verifiez si la detection automatique l'a agree.
3. Concluez : forcer la langue est-il une aide ou un biais ? Repondez en trois lignes en citant une exemple du tableau.

In [12]:
# Exercice 2 : langue forcee contre detection automatique
# Etape 1 : comparer la transcription automatique et la transcription forcee sur la source
# Etape 2 : juger la langue reelle a l'oreille et la confronter a la langue detectee
# Etape 3 : conclure en trois lignes, avec un exemple tire du tableau
print("TODO etudiant : comparer les deux transcriptions mot a mot")
print("TODO etudiant : conclure : forcer la langue aide-t-il ou biaise-t-il ?")

TODO etudiant : comparer les deux transcriptions mot a mot
TODO etudiant : conclure : forcer la langue aide-t-il ou biaise-t-il ?


### Exercice 3 : la robustesse du moteur au bruit

Un moteur de transcription se juge sur les signaux difficiles, pas sur les signaux propres.

1. Ajoutez un bruit blanc a l'extrait source a trois niveaux (par exemple 0 dB, 10 dB et 20 dB de rapport signal/bruit), mesurez avec `np.random.default_rng(seed)` pour rester reproductible.
2. Transcrivez chaque version bruitee, et mesurez le recouvrement de vocabulaire avec la transcription de reference du carnet (`normaliser(...)`).
3. A partir de quel niveau le moteur cesse-t-il de restituer le contenu ? Comparez avec Whisper small sur le meme signal bruite : lequel resiste le mieux ?

Cet exercice demande une GPU : la transcription bruitee de quatre signaux tient en quelques secondes sur le device 2.

In [13]:
# Exercice 3 : robustesse au bruit
# Etape 1 : bruit blanc a trois niveaux de rapport signal/bruit, avec une graine fixe
# Etape 2 : transcrire chaque version, mesurer le recouvrement de vocabulaire
# Etape 3 : comparer avec Whisper small sur le meme signal bruite
print("TODO etudiant : bruiter l'extrait source a trois niveaux de SNR")
print("TODO etudiant : transcrire et mesurer le recouvrement de vocabulaire")
print("TODO etudiant : comparer avec Whisper small, conclure sur la robustesse")

TODO etudiant : bruiter l'extrait source a trois niveaux de SNR
TODO etudiant : transcrire et mesurer le recouvrement de vocabulaire
TODO etudiant : comparer avec Whisper small, conclure sur la robustesse


## Synthese

- **La nomenclature Qwen3 est plus large que « Qwen3-Audio »** : elle separe en un modele omni-modal (Qwen3-Omni, 30 Md, qui comprend et parle) et un modele de reconnaissance vocale dedie (Qwen3-ASR, 1,7 Md et 0,6 Md). Le nom `Qwen/Qwen3-Audio` n'existe pas sur le hub, et le carnet **mesure** cette absence : la liste des modeles officiels Qwen pour le son est affichee par une cellule, pas affirmee dans un paragraphe.
- **Un moteur reel, execute pour de vrai** : `Qwen3-ASR-1.7B` (2,038 Md de parametres charges, 3,80 Go de VRAM alloues, 4,38 Go de pic, 9,4 s de chargement depuis le cache), sur le device 2 de la machine - identifie par son UUID, pas par une convention. Chaque transcription du carnet est la sortie du modele.
- **Comparer plutot que noter** : le depot ne documente pas le texte des extraits de `assets/`, donc il n'y a pas de reference contre laquelle noter. Trois moteurs independants sur le meme signal - Qwen3-ASR, Whisper small, Qwen2-Audio (mesure du carnet 02-13) - transforment l'incertitude en fait mesurable.
- **Le resultat central est negatif, et il est utile** : sur ce signal degrade, les trois moteurs, de 2023 a 2026, lisent le meme squelette et echouent aux memes endroits. Une generation plus recente ne repare pas un signal casse ; elle change la maniere dont on le **dit** - ici en donnant une identification de langue explicite, qui s'avere fausse (`Swedish` sur une source que la lecture forcee rend en francais).
- **L'accord entre moteurs ne prouve pas la justesse** : Whisper small (2023) et Qwen2-Audio (2024) se recouvrent a 0,500, contre 0,174 et 0,143 pour Qwen3-ASR (2026) - le moteur le plus recent est aussi le plus a part. Sans reference ecrite, on ne peut pas dire si c'est une amelioration ou un mode d'echec different ; on peut dire ou les moteurs s'accordent, et c'est deja beaucoup.
- **Les limites sont dites** : un seul signal de 4,9 s, pas de reference ecrite, un signal degrade, et le modele omni-modal de 30 Md reste a mesurer sur cette machine.

## References

Les sources de ce carnet ont ete archivees dans le gisement bibliographique partage, sous `G:\Mon Drive\MyIA\IA\Bibliographie IA` :

- **Qwen3-ASR Technical Report** - Qwen Team, arXiv 2601.21337 (30 janvier 2026) - `G:\Mon Drive\MyIA\IA\Bibliographie IA\Audio\2026 - Qwen Team - Qwen3-ASR Technical Report (arXiv 2601.21337).pdf`
- **Qwen3-Omni Technical Report** - Qwen Team, arXiv 2509.17765 (22 septembre 2025) - `G:\Mon Drive\MyIA\IA\Bibliographie IA\MachineLearning\Image Generation, Multimodality and Robustness\2025 - Qwen Team - Qwen3-Omni Technical Report (arXiv 2509.17765).pdf`
- **Qwen2-Audio Technical Report** - Yunfei Chu et al., arXiv 2407.10759 (15 juillet 2024) - `G:\Mon Drive\MyIA\IA\Bibliographie IA\Audio\2024 - Chu et al - Qwen2-Audio Technical Report (arXiv 2407.10759).pdf`
- **Carte de modele Qwen3-ASR** (capture de la page Hugging Face) - `G:\Mon Drive\MyIA\IA\Bibliographie IA\Audio\2026 - Qwen Team - Qwen3-ASR model card (capture Hugging Face) - source.md`

Les poids utilises sont ceux de `Qwen/Qwen3-ASR-1.7B` (Apache 2.0),charges par le package `qwen-asr`. Le temoin Whisper small est distribue par OpenAI sous licence MIT, via `faster-whisper`.

Le carnet 02-13 (PR #27714) reste la reference pour la generation Qwen2 et pour la lecture comparative avec Audio Flamingo (Kong et al., arXiv 2402.01831).